# Disagreement Direction Test for Sentiment Classification

This notebook demonstrates the **Disagreement Direction Test**: when a simple model (TF-IDF + Logistic Regression) and a complex model (character n-gram + RandomForest) disagree on a sentiment classification, does the *direction* of disagreement predict which model is correct?

**Hypothesis:** The simple model is more likely correct when it predicts the majority class (because it biases toward the mode), and the complex model is more likely correct when the simple model predicts the minority class (because the simple model is unreliable on rare patterns).

The notebook:
1. Loads pre-computed results from the full experiment (240 configurations across SST-2 and IMDB)
2. Runs a small-scale live experiment to demonstrate the methodology
3. Visualizes the results comparing the directional rule against three baselines

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.3')
_pip('datasets==4.0.0')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'tqdm==4.67.3')


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import gc
import json
import math
import os
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
from loguru import logger
from scipy import stats
from scipy.stats import binomtest
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

# Remove default loguru handler and add notebook-friendly one
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

1

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-2f324c-disagreement-asymmetry-oracle/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    local = Path("mini_demo_data.json")
    if local.exists():
        return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded data with {len(data['datasets'])} datasets")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")
print(f"Ablation table entries: {len(data['metadata']['ablation_table'])}")
print(f"Total configurations in full experiment: {data['metadata']['total_configurations']}")

Loaded data with 2 datasets
  sst2: 3 examples
  imdb: 3 examples
Ablation table entries: 24
Total configurations in full experiment: 240


In [5]:
# ============================================================================
# CONFIGURATION — Tunable parameters for the demo experiment
# ============================================================================
# Original full experiment values (commented out for reference):
#   DATASETS = ["sst2", "imdb"]
#   SAMPLE_SIZES = [100, 200, 500, 1000]
#   IMBALANCE_RATIOS = [1.0, 1.5, 2.0]
#   SEEDS = list(range(10))
#   Total: 2 datasets × 4 sizes × 3 ratios × 10 seeds = 240 configurations
#
# Demo values — minimal set that produces meaningful output:
DATASETS = ["sst2"]
SAMPLE_SIZES = [100, 200]
IMBALANCE_RATIOS = [1.0, 1.5]
SEEDS = [0, 1]
# Total demo: 1 × 2 × 2 × 2 = 8 configurations

print(f"Demo config: {len(DATASETS)} datasets × {len(SAMPLE_SIZES)} sizes × {len(IMBALANCE_RATIOS)} ratios × {len(SEEDS)} seeds = {len(DATASETS)*len(SAMPLE_SIZES)*len(IMBALANCE_RATIOS)*len(SEEDS)} configurations")

Demo config: 1 datasets × 2 sizes × 2 ratios × 2 seeds = 8 configurations


## Model Definitions

Two models are compared:
- **Simple model**: TF-IDF (unigram + bigram) + Logistic Regression
- **Complex model**: Character n-grams (3-5) + Random Forest (100 trees)

The simple model is expected to be reliable on common patterns but biased toward the majority class. The complex model can capture rare patterns but may overfit on small data.

In [6]:
def get_simple_model():
    """Simple model: TF-IDF + Logistic Regression."""
    return Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
        ('clf', LogisticRegression(max_iter=1000, random_state=42, n_jobs=1))
    ])


def get_complex_model():
    """Complex model: character n-grams + RandomForest."""
    return Pipeline([
        ('char_ngram', CountVectorizer(
            analyzer='char',
            ngram_range=(3, 5),
            max_features=20000
        )),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1))
    ])

## Data Loading and Preparation

Loads SST-2 or IMDB from HuggingFace, then creates a stratified subsample with controlled class imbalance. The `imbalance_ratio` parameter controls the majority:minority ratio (1.0 = balanced, 2.0 = 2:1 skew).

In [7]:
from datasets import load_dataset

def load_sst2():
    """Load SST-2 dataset."""
    logger.info("Loading SST-2 dataset")
    ds = load_dataset("stanfordnlp/sst2", split="train")
    texts = ds["sentence"]
    labels = ds["label"]
    del ds
    gc.collect()
    return texts, labels


def load_imdb():
    """Load IMDB dataset."""
    logger.info("Loading IMDB dataset")
    ds = load_dataset("stanfordnlp/imdb", split="train")
    texts = ds["text"]
    labels = ds["label"]
    del ds
    gc.collect()
    return texts, labels


def stratified_imbalanced_subsample(
    texts: list[str],
    labels: list[int],
    target_size: int,
    imbalance_ratio: float,
    random_state: int
) -> tuple[list[str], list[int]]:
    """Create a stratified subsample with controlled class imbalance.
    
    Args:
        texts: List of text samples
        labels: List of labels
        target_size: Total target sample size
        imbalance_ratio: Ratio of majority to minority class (1.0 = balanced)
        random_state: Random seed
    
    Returns:
        Subsampled texts and labels
    """
    rng = np.random.RandomState(random_state)

    # Separate by class
    class_0_idx = [i for i, l in enumerate(labels) if l == 0]
    class_1_idx = [i for i, l in enumerate(labels) if l == 1]

    # Calculate sizes based on imbalance ratio
    # imbalance_ratio = majority_size / minority_size
    # total = majority_size + minority_size
    minority_size = int(target_size / (1 + imbalance_ratio))
    majority_size = target_size - minority_size

    # Determine which class has more available samples
    if len(class_0_idx) >= len(class_1_idx):
        majority_class_idx = class_0_idx
        minority_class_idx = class_1_idx
    else:
        majority_class_idx = class_1_idx
        minority_class_idx = class_0_idx

    # Sample from each class
    sampled_majority = rng.choice(
        majority_class_idx,
        size=min(majority_size, len(majority_class_idx)),
        replace=False
    )
    sampled_minority = rng.choice(
        minority_class_idx,
        size=min(minority_size, len(minority_class_idx)),
        replace=False
    )

    # Combine and shuffle
    indices = np.concatenate([sampled_majority, sampled_minority])
    rng.shuffle(indices)

    return [texts[i] for i in indices], [labels[i] for i in indices]

## Baseline Methods

Three baselines are compared against the directional rule:
1. **Confidence baseline**: Pick the model with higher predicted probability
2. **Trust score baseline**: k-NN agreement in TF-IDF space (proxy for trust)
3. **Random baseline**: Random guessing (50/50)

In [8]:
def confidence_baseline(disagreements: list[dict]) -> list[int]:
    """Baseline: predict whichever model has higher confidence."""
    predictions = []
    for d in disagreements:
        if d['prob_simple'] > d['prob_complex']:
            predictions.append(d['pred_simple'])
        else:
            predictions.append(d['pred_complex'])
    return predictions


def trust_score_baseline(
    disagreements: list[dict],
    train_texts: list[str],
    train_labels: list[int]
) -> list[int]:
    """Baseline: trust-score style proxy using nearest-neighbor agreement in TF-IDF space."""
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    try:
        train_tfidf = vectorizer.fit_transform(train_texts)
        nn = NearestNeighbors(n_neighbors=5, metric='cosine')
        nn.fit(train_tfidf)

        predictions = []
        for d in disagreements:
            test_vec = vectorizer.transform([d['text']])
            _, indices = nn.kneighbors(test_vec)
            neighbor_labels = [train_labels[i] for i in indices[0]]
            majority_label = Counter(neighbor_labels).most_common(1)[0][0]
            predictions.append(majority_label)
    except Exception as e:
        logger.warning(f"Trust score baseline failed: {e}")
        # Fallback to random
        rng = np.random.RandomState(42)
        predictions = rng.randint(0, 2, size=len(disagreements)).tolist()

    return predictions


def random_baseline(n: int) -> list[int]:
    """Random baseline."""
    return np.random.RandomState(42).randint(0, 2, size=n).tolist()

## Statistical Tests

Two statistical tests are used:
1. **Binomial test**: Tests if the directional rule accuracy exceeds 50% (chance)
2. **McNemar's test**: Compares paired predictions between the directional rule and each baseline

In [9]:
def binomial_test_on_directional_rule(
    disagreements: list[dict],
    majority_class: int
) -> tuple[float, float]:
    """Test if directional rule accuracy > 50% using binomial test."""
    correct = sum(
        1 for d in disagreements
        if (d['pred_simple'] == majority_class and d['pred_simple'] == d['true_label']) or
           (d['pred_simple'] != majority_class and d['pred_complex'] == d['true_label'])
    )
    n = len(disagreements)
    if n == 0:
        return 0.0, 1.0
    result = binomtest(correct, n, 0.5, alternative='greater')
    p_value = result.pvalue
    accuracy = correct / n
    return accuracy, p_value


def mcnemar_test(
    y_true: list[int],
    pred1: list[int],
    pred2: list[int]
) -> tuple[float, float]:
    """McNemar's test between two prediction sets."""
    a = sum(1 for yt, p1, p2 in zip(y_true, pred1, pred2)
            if p1 == yt and p2 != yt)  # pred1 correct, pred2 wrong
    b = sum(1 for yt, p1, p2 in zip(y_true, pred1, pred2)
            if p1 != yt and p2 == yt)  # pred1 wrong, pred2 correct
    d = sum(1 for yt, p1, p2 in zip(y_true, pred1, pred2)
            if p1 == yt and p2 == yt)  # both correct

    if a + b == 0:
        return 0.0, 1.0

    if a + b < 25:
        result = binomtest(a, a + b, 0.5, alternative='two-sided')
        p_value = result.pvalue
    else:
        chi2 = (abs(a - b) - 1) ** 2 / (a + b)
        p_value = stats.chi2.sf(chi2, 1)

    accuracy1 = (a + d) / len(y_true) if len(y_true) > 0 else 0
    accuracy2 = (b + d) / len(y_true) if len(y_true) > 0 else 0

    return accuracy1, p_value

## Experiment Execution

The core experiment function: subsamples data with controlled imbalance, trains both models on 80% of the data, evaluates on 20%, identifies disagreements, and compares the directional rule against all baselines.

In [10]:
def run_single_experiment(
    dataset_name: str,
    texts: list[str],
    labels: list[int],
    sample_size: int,
    imbalance_ratio: float,
    seed: int
) -> dict[str, Any]:
    """Run a single experiment configuration."""
    start_time = time.time()
    logger.info(
        f"Running: dataset={dataset_name}, size={sample_size}, "
        f"imbalance={imbalance_ratio}, seed={seed}"
    )

    # Subsample
    sub_texts, sub_labels = stratified_imbalanced_subsample(
        texts, labels, sample_size, imbalance_ratio, seed
    )

    # Determine majority class
    label_counts = Counter(sub_labels)
    majority_class = label_counts.most_common(1)[0][0]
    logger.debug(f"Majority class: {majority_class}, distribution: {label_counts}")

    # Train/test split (80/20), stratified
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        sub_texts, sub_labels, test_size=0.2, random_state=seed, stratify=sub_labels
    )

    # Fit simple model
    simple_model = get_simple_model()
    simple_model.fit(train_texts, train_labels)
    simple_preds = simple_model.predict(test_texts)
    simple_probs = simple_model.predict_proba(test_texts)

    # Fit complex model
    complex_model = get_complex_model()
    complex_model.fit(train_texts, train_labels)
    complex_preds = complex_model.predict(test_texts)
    complex_probs = complex_model.predict_proba(test_texts)

    # Find disagreements
    disagreements = []
    for i, (text, true_label) in enumerate(zip(test_texts, test_labels)):
        if simple_preds[i] != complex_preds[i]:
            disagreements.append({
                'text': text,
                'true_label': int(true_label),
                'pred_simple': int(simple_preds[i]),
                'pred_complex': int(complex_preds[i]),
                'prob_simple': float(max(simple_probs[i])),
                'prob_complex': float(max(complex_probs[i])),
                'sample_index': i
            })

    logger.debug(
        f"Found {len(disagreements)} disagreements out of {len(test_texts)} test samples"
    )

    # Compute baselines on disagreement set
    if len(disagreements) > 0:
        # Random baseline
        random_preds = random_baseline(len(disagreements))

        # Confidence baseline
        conf_preds = confidence_baseline(disagreements)

        # Trust score baseline
        trust_preds = trust_score_baseline(disagreements, train_texts, train_labels)

        # Directional rule
        dir_preds = []
        for d in disagreements:
            if d['pred_simple'] == majority_class:
                dir_preds.append(d['pred_simple'])
            else:
                dir_preds.append(d['pred_complex'])

        # True labels for disagreements
        true_labels_d = [d['true_label'] for d in disagreements]

        # Compute metrics
        dir_accuracy = accuracy_score(true_labels_d, dir_preds)
        conf_accuracy = accuracy_score(true_labels_d, conf_preds)
        trust_accuracy = accuracy_score(true_labels_d, trust_preds)
        random_accuracy = accuracy_score(true_labels_d, random_preds)

        # Statistical tests
        _, dir_pvalue = binomial_test_on_directional_rule(disagreements, majority_class)

        # McNemar's tests between directional rule and baselines
        _, mcnemar_conf_pvalue = mcnemar_test(true_labels_d, dir_preds, conf_preds)
        _, mcnemar_trust_pvalue = mcnemar_test(true_labels_d, dir_preds, trust_preds)
        _, mcnemar_random_pvalue = mcnemar_test(true_labels_d, dir_preds, random_preds)

    else:
        # No disagreements — use None/NaN to signal "not applicable"
        dir_accuracy = conf_accuracy = trust_accuracy = random_accuracy = float('nan')
        dir_pvalue = mcnemar_conf_pvalue = mcnemar_trust_pvalue = mcnemar_random_pvalue = float('nan')
        dir_preds = conf_preds = trust_preds = random_preds = []

    # Overall accuracies
    simple_accuracy = accuracy_score(test_labels, simple_preds)
    complex_accuracy = accuracy_score(test_labels, complex_preds)

    elapsed = time.time() - start_time
    logger.info(
        f"Completed in {elapsed:.1f}s: size={sample_size}, seed={seed}, "
        f"disagreements={len(disagreements)}, dir_acc={dir_accuracy:.3f}"
    )

    return {
        'dataset': dataset_name,
        'sample_size': sample_size,
        'imbalance_ratio': imbalance_ratio,
        'seed': seed,
        'train_size': len(train_texts),
        'test_size': len(test_texts),
        'simple_accuracy': float(simple_accuracy),
        'complex_accuracy': float(complex_accuracy),
        'num_disagreements': len(disagreements),
        'disagreement_ratio': float(len(disagreements) / len(test_texts)) if len(test_texts) > 0 else 0.0,
        'majority_class': int(majority_class),
        'class_distribution': {str(k): int(v) for k, v in label_counts.items()},
        'directional_rule_accuracy': dir_accuracy,
        'confidence_baseline_accuracy': conf_accuracy,
        'trust_score_baseline_accuracy': trust_accuracy,
        'random_baseline_accuracy': random_accuracy,
        'directional_rule_pvalue': dir_pvalue,
        'mcnemar_vs_confidence_pvalue': mcnemar_conf_pvalue,
        'mcnemar_vs_trust_score_pvalue': mcnemar_trust_pvalue,
        'mcnemar_vs_random_pvalue': mcnemar_random_pvalue,
        'elapsed_seconds': float(elapsed),
    }


def _safe_mean(values: list[float]) -> float:
    """Compute mean ignoring NaN values; return NaN if all are NaN."""
    clean = [v for v in values if not math.isnan(v)]
    if not clean:
        return float('nan')
    return float(np.mean(clean))


def _safe_std(values: list[float]) -> float:
    """Compute std ignoring NaN values; return NaN if fewer than 2 valid."""
    clean = [v for v in values if not math.isnan(v)]
    if len(clean) < 2:
        return float('nan')
    return float(np.std(clean))


def generate_ablation_table(results: list[dict]) -> dict[str, Any]:
    """Generate ablation summary table."""
    summary = {}
    for dataset in set(r['dataset'] for r in results if 'error' not in r):
        dataset_results = [r for r in results if r.get('dataset') == dataset and 'error' not in r]
        for sample_size in set(r['sample_size'] for r in dataset_results):
            for imbalance_ratio in set(r['imbalance_ratio'] for r in dataset_results):
                config_results = [
                    r for r in dataset_results
                    if r['sample_size'] == sample_size
                    and r['imbalance_ratio'] == imbalance_ratio
                ]
                if not config_results:
                    continue
                with_disagreements = [r for r in config_results if r['num_disagreements'] > 0]
                key = f"{dataset}_size{sample_size}_imb{imbalance_ratio}"
                summary[key] = {
                    'dataset': dataset,
                    'sample_size': sample_size,
                    'imbalance_ratio': imbalance_ratio,
                    'n_seeds': len(config_results),
                    'n_with_disagreements': len(with_disagreements),
                    'mean_directional_accuracy': _safe_mean([r['directional_rule_accuracy'] for r in config_results]),
                    'std_directional_accuracy': _safe_std([r['directional_rule_accuracy'] for r in config_results]),
                    'mean_confidence_accuracy': _safe_mean([r['confidence_baseline_accuracy'] for r in config_results]),
                    'mean_trust_score_accuracy': _safe_mean([r['trust_score_baseline_accuracy'] for r in config_results]),
                    'mean_random_accuracy': _safe_mean([r['random_baseline_accuracy'] for r in config_results]),
                    'mean_disagreement_ratio': float(np.mean([r['disagreement_ratio'] for r in config_results])),
                    'mean_directional_pvalue': _safe_mean([r['directional_rule_pvalue'] for r in config_results]),
                }
    return summary

## Run the Demo Experiment

Now we run the experiment with the demo configuration. Each configuration trains both models and evaluates the directional rule. This takes a few minutes for the demo set (8 configurations).

In [11]:
def run_demo_experiments():
    """Run experiments with the demo configuration."""
    all_results = []

    for dataset_name in DATASETS:
        logger.info(f"=== Processing dataset: {dataset_name} ===")
        if dataset_name == "sst2":
            texts, labels = load_sst2()
        else:
            texts, labels = load_imdb()

        for sample_size in SAMPLE_SIZES:
            for imbalance_ratio in IMBALANCE_RATIOS:
                for seed in SEEDS:
                    try:
                        result = run_single_experiment(
                            dataset_name, texts, labels,
                            sample_size, imbalance_ratio, seed
                        )
                        all_results.append(result)
                    except Exception as e:
                        logger.error(f"Experiment failed: {e}")
                        all_results.append({
                            'dataset': dataset_name,
                            'sample_size': sample_size,
                            'imbalance_ratio': imbalance_ratio,
                            'seed': seed,
                            'error': str(e),
                        })

        del texts, labels
        gc.collect()
        logger.info(f"Finished {dataset_name}")

    logger.info(f"Completed {len(all_results)} experiments total")
    return all_results


# Run the experiments
results = run_demo_experiments()

# Generate ablation table
ablation = generate_ablation_table(results)

print(f"\nCompleted {len(results)} configurations")
valid_results = [r for r in results if 'error' not in r]
with_dis = [r for r in valid_results if r['num_disagreements'] > 0]
print(f"Configurations with disagreements: {len(with_dis)}")
print(f"Total disagreements analyzed: {sum(r['num_disagreements'] for r in with_dis)}")

08:09:48|INFO   |=== Processing dataset: sst2 ===


08:09:48|INFO   |Loading SST-2 dataset


README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

08:09:59|INFO   |Running: dataset=sst2, size=200, imbalance=1.5, seed=1


08:10:00|ERROR  |Experiment failed: Wrong key type: '45795' of type '<class 'numpy.int64'>'. Expected one of int, slice, range, str or Iterable.


08:10:00|INFO   |Finished sst2


08:10:00|INFO   |Completed 8 experiments total



Completed 8 configurations
Configurations with disagreements: 0
Total disagreements analyzed: 0


## Results and Visualization

Visualize the results: compare the directional rule against the three baselines across all configurations, and show the full ablation table from the original experiment.

In [12]:
# ============================================================================
# VISUALIZATION
# ============================================================================

# --- 1. Per-configuration results table ---
print("="*80)
print("DEMO EXPERIMENT RESULTS")
print("="*80)

valid_results = [r for r in results if 'error' not in r]
rows = []
for r in valid_results:
    dir_acc = f"{r['directional_rule_accuracy']:.3f}" if not math.isnan(r['directional_rule_accuracy']) else "N/A"
    conf_acc = f"{r['confidence_baseline_accuracy']:.3f}" if not math.isnan(r['confidence_baseline_accuracy']) else "N/A"
    trust_acc = f"{r['trust_score_baseline_accuracy']:.3f}" if not math.isnan(r['trust_score_baseline_accuracy']) else "N/A"
    rand_acc = f"{r['random_baseline_accuracy']:.3f}" if not math.isnan(r['random_baseline_accuracy']) else "N/A"
    rows.append({
        'Dataset': r['dataset'],
        'Size': r['sample_size'],
        'Imb.': r['imbalance_ratio'],
        'Seed': r['seed'],
        'Disagreements': r['num_disagreements'],
        'Simple Acc': f"{r['simple_accuracy']:.3f}",
        'Complex Acc': f"{r['complex_accuracy']:.3f}",
        'Dir. Rule': dir_acc,
        'Conf. Base': conf_acc,
        'Trust Base': trust_acc,
        'Random': rand_acc,
    })

df_demo = pd.DataFrame(rows)
print(df_demo.to_string(index=False))

# --- 2. Overall summary ---
with_dis = [r for r in valid_results if r['num_disagreements'] > 0]
if len(with_dis) > 0:
    print(f"\n{'='*80}")
    print("OVERALL SUMMARY (demo)")
    print(f"{'='*80}")
    print(f"Mean directional rule accuracy:  {_safe_mean([r['directional_rule_accuracy'] for r in with_dis]):.3f}")
    print(f"Mean confidence baseline acc:    {_safe_mean([r['confidence_baseline_accuracy'] for r in with_dis]):.3f}")
    print(f"Mean trust score baseline acc:   {_safe_mean([r['trust_score_baseline_accuracy'] for r in with_dis]):.3f}")
    print(f"Mean random baseline acc:        {_safe_mean([r['random_baseline_accuracy'] for r in with_dis]):.3f}")

# --- 3. Bar chart: demo results ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Accuracy comparison per configuration
ax1 = axes[0]
if len(with_dis) > 0:
    x = np.arange(len(with_dis))
    width = 0.2
    labels = [f"{r['dataset'][:3]}-{r['sample_size']}-{r['seed']}" for r in with_dis]
    
    dir_vals = [r['directional_rule_accuracy'] for r in with_dis]
    conf_vals = [r['confidence_baseline_accuracy'] for r in with_dis]
    trust_vals = [r['trust_score_baseline_accuracy'] for r in with_dis]
    rand_vals = [r['random_baseline_accuracy'] for r in with_dis]
    
    ax1.bar(x - 1.5*width, dir_vals, width, label='Directional Rule', color='#2196F3')
    ax1.bar(x - 0.5*width, conf_vals, width, label='Confidence Base', color='#4CAF50')
    ax1.bar(x + 0.5*width, trust_vals, width, label='Trust Score Base', color='#FF9800')
    ax1.bar(x + 1.5*width, rand_vals, width, label='Random Base', color='#9E9E9E')
    
    ax1.set_xlabel('Configuration')
    ax1.set_ylabel('Accuracy on Disagreements')
    ax1.set_title('Directional Rule vs Baselines (Demo)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=45, ha='right')
    ax1.legend(fontsize=8)
    ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Chance (0.5)')
    ax1.set_ylim(0, 1.1)
    ax1.grid(axis='y', alpha=0.3)

# Right: Ablation table from full experiment
ax2 = axes[1]
ablation_data = data['metadata']['ablation_table']
ablation_keys = list(ablation_data.keys())
ablation_labels = [k.replace('_', ' ').replace('size', 'n=').replace('imb', 'imb=').title() for k in ablation_keys]
dir_accs = [ablation_data[k]['mean_directional_accuracy'] for k in ablation_keys]
conf_accs = [ablation_data[k]['mean_confidence_accuracy'] for k in ablation_keys]
trust_accs = [ablation_data[k]['mean_trust_score_accuracy'] for k in ablation_keys]
rand_accs = [ablation_data[k]['mean_random_accuracy'] for k in ablation_keys]

x2 = np.arange(len(ablation_keys))
width2 = 0.2
ax2.bar(x2 - 1.5*width2, dir_accs, width2, label='Directional Rule', color='#2196F3')
ax2.bar(x2 - 0.5*width2, conf_accs, width2, label='Confidence Base', color='#4CAF50')
ax2.bar(x2 + 0.5*width2, trust_accs, width2, label='Trust Score Base', color='#FF9800')
ax2.bar(x2 + 1.5*width2, rand_accs, width2, label='Random Base', color='#9E9E9E')

ax2.set_xlabel('Configuration (Full Experiment)')
ax2.set_ylabel('Mean Accuracy on Disagreements')
ax2.set_title('Ablation: Directional Rule vs Baselines (240 Configs)')
ax2.set_xticks(x2)
ax2.set_xticklabels(ablation_labels, rotation=90, fontsize=6)
ax2.legend(fontsize=7)
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Chance (0.5)')
ax2.set_ylim(0, 1.1)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved to results.png")

# --- 4. Full ablation table ---
print(f"\n{'='*80}")
print("FULL ABLATION TABLE (from 240 configurations)")
print(f"{'='*80}")

ablation_rows = []
for key, val in ablation_data.items():
    ablation_rows.append({
        'Dataset': val['dataset'],
        'Size': val['sample_size'],
        'Imbalance': val['imbalance_ratio'],
        'Seeds': val['n_seeds'],
        'With Disagreements': val['n_with_disagreements'],
        'Mean Dir. Acc': f"{val['mean_directional_accuracy']:.3f}" if not math.isnan(val['mean_directional_accuracy']) else 'N/A',
        'Std Dir. Acc': f"{val['std_directional_accuracy']:.3f}" if not math.isnan(val['std_directional_accuracy']) else 'N/A',
        'Mean Conf. Acc': f"{val['mean_confidence_accuracy']:.3f}" if not math.isnan(val['mean_confidence_accuracy']) else 'N/A',
        'Mean Trust Acc': f"{val['mean_trust_score_accuracy']:.3f}" if not math.isnan(val['mean_trust_score_accuracy']) else 'N/A',
        'Mean Random Acc': f"{val['mean_random_accuracy']:.3f}" if not math.isnan(val['mean_random_accuracy']) else 'N/A',
        'Mean Disagree Ratio': f"{val['mean_disagreement_ratio']:.3f}",
        'Mean p-value': f"{val['mean_directional_pvalue']:.3f}" if not math.isnan(val['mean_directional_pvalue']) else 'N/A',
    })

df_ablation = pd.DataFrame(ablation_rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(df_ablation.to_string(index=False))

# --- 5. Key finding ---
print(f"\n{'='*80}")
print("KEY FINDING")
print(f"{'='*80}")
all_dir = [ablation_data[k]['mean_directional_accuracy'] for k in ablation_keys if not math.isnan(ablation_data[k]['mean_directional_accuracy'])]
all_conf = [ablation_data[k]['mean_confidence_accuracy'] for k in ablation_keys if not math.isnan(ablation_data[k]['mean_confidence_accuracy'])]
all_trust = [ablation_data[k]['mean_trust_score_accuracy'] for k in ablation_keys if not math.isnan(ablation_data[k]['mean_trust_score_accuracy'])]
print(f"Overall mean directional rule accuracy:  {np.mean(all_dir):.3f}")
print(f"Overall mean confidence baseline acc:    {np.mean(all_conf):.3f}")
print(f"Overall mean trust score baseline acc:   {np.mean(all_trust):.3f}")
print(f"\nThe directional rule does NOT consistently outperform the baselines,")
print(f"suggesting the directional hypothesis does not hold on sentiment classification.")

DEMO EXPERIMENT RESULTS
Empty DataFrame
Columns: []
Index: []



Chart saved to results.png

FULL ABLATION TABLE (from 240 configurations)
Dataset  Size  Imbalance  Seeds  With Disagreements Mean Dir. Acc Std Dir. Acc Mean Conf. Acc Mean Trust Acc Mean Random Acc Mean Disagree Ratio Mean p-value
   sst2   100        1.0     10                  10         0.486        0.208          0.479          0.468           0.494               0.335        0.596
   sst2   100        1.5     10                   6         0.333        0.373          0.583          0.583           0.500               0.050        0.833
   sst2   100        2.0     10                   4         0.625        0.415          0.625          0.875           0.250               0.025        0.688
   sst2   200        1.0     10                  10         0.444        0.183          0.478          0.500           0.443               0.367        0.647
   sst2   200        1.5     10                   9         0.431        0.294          0.533          0.470           0.563           